## Datasets


We explore four real-world datasets in this notebook:

- `diamonds` — physical attributes and prices of diamonds (used for most examples).
- `flights` — NYC flight departures (used to show how *missing* values can be informative).
- `mpg` — fuel economy of popular car models (introduced later, in the covariation section).
- `presidential` — terms of recent US presidents (used for date-based timeline plots).


In [ ]:
import pandas as pd
import seaborn as sns


### The `diamonds` dataset

`diamonds` ships with seaborn and contains ~53,940 round-cut diamonds described by 10 variables ([ggplot2 docs](https://ggplot2.tidyverse.org/reference/diamonds.html)):

- `price` — price in US dollars ($326–$18,823).
- `carat` — weight of the diamond (0.2–5.01).
- `cut` — quality of the cut, **ordered**: Fair < Good < Very Good < Premium < Ideal.
- `color` — diamond color, from D (best) to J (worst).
- `clarity` — how clear the diamond is, **ordered**: I1 (worst) < SI2 < SI1 < VS2 < VS1 < VVS2 < VVS1 < IF (best).
- `x` — length in mm (0–10.74).
- `y` — width in mm (0–58.9).
- `z` — depth in mm (0–31.8).
- `depth` — total depth percentage, `2 * z / (x + y)` (43–79).
- `table` — width of the top of the diamond relative to its widest point (43–95).

The ordered quality grades (`cut`, `color`, `clarity`) make this dataset ideal for asking how values vary within a variable and across categories.


![Illustration for Diamonds Dataset](../assets/03/diamonds.png)

In [ ]:
diamonds = sns.load_dataset("diamonds")
diamonds

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...
53935,0.72,Ideal,D,SI1,60.8,57.0,2757,5.75,5.76,3.50
53936,0.72,Good,D,SI1,63.1,55.0,2757,5.69,5.75,3.61
53937,0.70,Very Good,D,SI1,62.8,60.0,2757,5.66,5.68,3.56
53938,0.86,Premium,H,SI2,61.0,58.0,2757,6.15,6.12,3.74


### The `flights` dataset

`flights` comes from R's `nycflights13` package (via the [Rdatasets](https://vincentarelbundock.github.io/Rdatasets/doc/nycflights13/flights.html) mirror) and holds on-time records for all 336,776 flights that departed the NYC airports (JFK, LGA, EWR) in 2013:

- `year`, `month`, `day` — date of departure.
- `dep_time`, `arr_time` — actual departure and arrival times (format HHMM or HMM, local tz).
- `sched_dep_time`, `sched_arr_time` — scheduled departure and arrival times (HHMM or HMM, local tz).
- `dep_delay`, `arr_delay` — departure and arrival delays in minutes; negative values mean the flight was early.
- `carrier` — two-letter carrier abbreviation.
- `flight` — flight number.
- `tailnum` — plane tail number.
- `origin`, `dest` — origin and destination airports.
- `air_time` — time spent in the air, in minutes.
- `distance` — distance between airports, in miles.
- `hour`, `minute` — scheduled departure time split into hour and minute.
- `time_hour` — scheduled date and hour of the flight.

We read it straight from a CSV URL and drop the `rownames` index column added by the R export. A missing `dep_time` marks a **cancelled** flight — a detail we use later to show how missingness can itself be meaningful.


In [ ]:

FLIGHTS_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/nycflights13/flights.csv"
flights = pd.read_csv(FLIGHTS_URL).drop(columns="rownames")
flights

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336771,2013,9,30,NaN,1455,NaN,NaN,1634,NaN,9E,3393,NaN,JFK,DCA,NaN,213,14,55,2013-09-30T18:00:00Z
336772,2013,9,30,NaN,2200,NaN,NaN,2312,NaN,9E,3525,NaN,LGA,SYR,NaN,198,22,0,2013-10-01T02:00:00Z
336773,2013,9,30,NaN,1210,NaN,NaN,1330,NaN,MQ,3461,N535MQ,LGA,BNA,NaN,764,12,10,2013-09-30T16:00:00Z
336774,2013,9,30,NaN,1159,NaN,NaN,1344,NaN,MQ,3572,N511MQ,LGA,CLE,NaN,419,11,59,2013-09-30T15:00:00Z


### The `mpg` dataset

`mpg` ships with ggplot2 (via the [Rdatasets](https://vincentarelbundock.github.io/Rdatasets/doc/ggplot2/mpg.html) mirror) and holds a 234-row, 11-variable subset of the EPA's [fuel-economy data](https://fueleconomy.gov/). It keeps only the 38 car models that had a new release every year between 1999 and 2008 — a proxy for popularity:

- `manufacturer` — manufacturer name.
- `model` — model name.
- `displ` — engine displacement, in litres.
- `year` — year of manufacture.
- `cyl` — number of cylinders.
- `trans` — type of transmission.
- `drv` — drive train: `f` = front-wheel, `r` = rear-wheel, `4` = 4wd.
- `cty` — city miles per gallon.
- `hwy` — highway miles per gallon.
- `fl` — fuel type.
- `class` — "type" of car (e.g. compact, midsize, SUV).

As with `flights`, we read it straight from a CSV URL and drop the `rownames` index column added by the R export. The unordered `class` categories make it a good fit for reordering by a summary statistic.

In [ ]:
MPG_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/ggplot2/mpg.csv"

mpg = pd.read_csv(MPG_URL).drop(columns="rownames")
mpg.head()

,manufacturer,model,displ,year,cyl,trans,drv,cty,hwy,fl,class
0,audi,a4,1.8,1999,4,auto(l5),f,18,29,p,compact
1,audi,a4,1.8,1999,4,manual(m5),f,21,29,p,compact
2,audi,a4,2.0,2008,4,manual(m6),f,20,31,p,compact
3,audi,a4,2.0,2008,4,auto(av),f,21,30,p,compact
4,audi,a4,2.8,1999,6,auto(l5),f,16,26,p,compact


### The `presidential` dataset

`presidential` ships with ggplot2 (via the [Rdatasets](https://vincentarelbundock.github.io/Rdatasets/doc/ggplot2/presidential.html) mirror) and lists the terms of the 12 US presidents from Eisenhower to Trump in 4 variables. The data is in the public domain:

- `name` — last name of the president.
- `start` — start date of the presidency.
- `end` — end date of the presidency.
- `party` — party of the president (`Democratic` or `Republican`).

As with `flights` and `mpg`, we read it straight from a CSV URL and drop the `rownames` index column added by the R export. We also parse `start` and `end` as dates so the terms can be drawn on a time axis. The date ranges and two-category `party` make it a natural fit for timeline plots.

In [ ]:
PRESIDENTIAL_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/ggplot2/presidential.csv"

presidential = pd.read_csv(PRESIDENTIAL_URL, parse_dates=["start", "end"]).drop(columns="rownames")
presidential